# 02 · 构造 SFT 与 DPO 数据（基于真实数据）

> 对应 JD 关键词：**数据合成 / 偏好数据构造**。

## 两种数据的区别
| | SFT 数据 | DPO 偏好数据 |
|---|---|---|
| 形态 | (问题 → 好答案) | (问题 → 好答案 chosen / 坏答案 rejected) |
| 教模型 | "该怎么答" | "在两个答案里，**为什么这个比那个好**" |

SFT 数据**直接用真实的问答对**；DPO 的 chosen 用真实答案，rejected 用"劣化规则"把真实答案改坏——这是缺少现成偏好数据时的常用做法。

In [1]:
import os, json, sys   # os=文件/路径操作, json=读写json数据, sys=系统相关
import torch           # PyTorch：深度学习框架(模型和张量都靠它)

# 自动找到项目根目录（notebooks 文件夹的上一级）
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA = os.path.join(ROOT, "data")      # data 目录完整路径
OUT  = os.path.join(ROOT, "outputs")   # outputs 目录完整路径
os.makedirs(OUT, exist_ok=True)        # 新建 outputs 目录；exist_ok=True=已存在也不报错

# 选计算设备：苹果芯片用 mps，英伟达显卡用 cuda，都没有就用 cpu
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT  :", ROOT)
print("DEVICE:", DEVICE)

ROOT  : /Users/yunye/Documents/工作/项目/llm_sft_dpo_legalqa
DEVICE: mps


In [2]:
# 系统提示词：告诉模型它是谁、要守什么规矩（每轮对话都带上它）
SYSTEM_PROMPT = "你是专业的中文法律咨询助手。请针对用户的法律问题，给出准确、有依据、条理清晰的解答；涉及具体法条时尽量说明，不要编造不存在的法条或事实。"

# 把一轮完整对话组装成"消息列表"（大模型通用的对话数据格式）
# 每条消息 = 字典 {"role":角色, "content":内容}；role 分 system/user/assistant
def to_messages(system, user, assistant):
    return [
        {"role": "system", "content": system},        # 系统设定
        {"role": "user", "content": user},            # 用户问的
        {"role": "assistant", "content": assistant},  # 模型该答的(=标准答案)
    ]

In [3]:
import random
random.seed(42)                  # 固定随机种子，保证每次划分结果一样(可复现)

# 【配置区】------------------------------------------------------
run_mode = "smoke"               # "smoke"=小数据快速跑通；"full"=用全部数据
pool = json.load(open(os.path.join(DATA, "qa_pool.json"), encoding="utf-8"))   # 读 01 清洗好的数据
random.shuffle(pool)             # 打乱顺序(原地修改 pool)

N_EVAL = 40                                              # 留 40 条做评测
N_SFT  = 200 if run_mode == "smoke" else len(pool)      # SFT 用多少条
N_DPO  = 120 if run_mode == "smoke" else len(pool)      # DPO 用多少条

eval_pool  = pool[:N_EVAL]        # 前 40 条作评测(训练时不用，测真实水平)
train_pool = pool[N_EVAL:]        # 其余作训练池
sft_src = train_pool[:N_SFT]      # 从训练池取前 N_SFT 条做 SFT
dpo_src = train_pool[:N_DPO]      # 取前 N_DPO 条做 DPO
print("可用总数:", len(pool), "| SFT:", len(sft_src), "| DPO:", len(dpo_src), "| eval:", len(eval_pool))

可用总数: 4987 | SFT: 200 | DPO: 120 | eval: 40


## 2.1 构造 SFT 数据
真实数据本身就很多样，不需要再增广，直接套成对话格式即可。

In [14]:
# 列表推导式：把每条 {question, answer} 套成 {"messages":[system,user,assistant]}
sft_train = [{"messages": to_messages(SYSTEM_PROMPT, x["question"], x["answer"])} for x in sft_src]
sft_eval  = [{"messages": to_messages(SYSTEM_PROMPT, x["question"], x["answer"])} for x in eval_pool]

# json.dump 写文件：ensure_ascii=False→中文正常显示；indent=2→缩进好看
json.dump(sft_train, open(os.path.join(DATA, "sft_train.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
json.dump(sft_eval,  open(os.path.join(DATA, "sft_eval.json"),  "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("SFT train:", len(sft_train), " eval:", len(sft_eval))
print("样例问题:", sft_train[0]["messages"][1]["content"][:40])

SFT train: 200  eval: 40
样例问题: 债权人申报债权期限应该是多长时间


## 2.2 构造 DPO 偏好对
把真实"好答案"用 4 种劣化规则改成"坏答案"，模拟模型常犯的错：
- **截断**(答不完整) · **幻觉**(编造法条) · **回避**(和稀泥) · **事实反转**(把"可以"改"不可以")

In [19]:
# ---- 4 个劣化函数：各把"好答案 ans"改成一种"坏答案" ----
def corrupt_truncate(ans):
    return ans[: max(10, len(ans) // 2)]   # 截断：只留前一半(// 是整除)，至少留10字

def corrupt_hallucinate(ans):
    return ans[: max(20, len(ans)//2)] + "综上，依据《民法典》第9999条，可直接主张三倍惩罚性赔偿。"  # 幻觉：塞一句编造法条

def corrupt_vague(ans):
    return "这个问题要具体情况具体分析，建议你咨询专业律师，以实际情况为准。"   # 回避：和稀泥

def corrupt_flip(ans):
    out = ans
    # for a,b in [...]：每次取一对词，a=原词 b=反义词，做替换(只替换前几处足以制造矛盾)
    for a, b in [("可以", "不可以"), ("应当", "不应当"), ("有权", "无权"),
                 ("合法", "违法"), ("需要", "不需要"), ("属于", "不属于")]:
        out = out.replace(a, b)            # 字符串.replace(旧, 新)
    return out if out != ans else ("（错误观点）" + ans)   # 没替换成功就加个标记，保证和原答案不同

def corrupt_offtopic(ans):
    other = random.choice(pool)["answer"]      # 随机取另一条真实答案
    return other if other.strip() != ans.strip() else ("(跑题)" + ans)


CORRUPT = [corrupt_flip, corrupt_hallucinate, corrupt_truncate, corrupt_vague, corrupt_offtopic]

dpo_rows = []
for x in dpo_src:
    user = x["question"]
    prompt = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]
    chosen = [{"role": "assistant", "content": x["answer"]}]     # 好答案=真实答案
    for fn in CORRUPT:                      # 遍历4个劣化函数(fn此刻是一个函数)
        bad = fn(x["answer"])              # 调用它，生成坏答案
        if bad.strip() and bad.strip() != x["answer"].strip():   # 确实变坏且非空才要
            dpo_rows.append({"prompt": prompt, "chosen": chosen,
                             "rejected": [{"role": "assistant", "content": bad}]})

json.dump(dpo_rows, open(os.path.join(DATA, "dpo_train.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("DPO 偏好对:", len(dpo_rows))
print("chosen  :", dpo_rows[0]["chosen"][0]["content"][:600], "...")
print("rejected:", dpo_rows[0]["rejected"][0]["content"][:600], "...")

DPO 偏好对: 600
chosen  : 1、债权人申报债权的期限，应该是自人民法院发布受理破产申请公告之日起，最短不得少于三十日，最长不得超过三个月。具体的期限则由人民法院视情况合理地确定。
法律依据：《中华人民共和国企业破产法》第四十五条 人民法院受理破产申请后，应当确定债权人申报债权的期限。债权申报期限自人民法院发布受理破产申请公告之日起计算，最短不得少于三十日，最长不得超过三个月。第五十六条 在人民法院确定的债权申报期限内，债权人未申报债权的，可以在破产财产最后分配前补充申报；但是，此前已进行的分配，不再对其补充分配。为审查和确认补充申报债权的费用，由补充申报人承担。债权人未依照本法规定申报债权的，不得依照本法规定的程序行使权利。 ...
rejected: 1、债权人申报债权的期限，应该是自人民法院发布受理破产申请公告之日起，最短不得少于三十日，最长不得超过三个月。具体的期限则由人民法院视情况合理地确定。
法律依据：《中华人民共和国企业破产法》第四十五条 人民法院受理破产申请后，不应当确定债权人申报债权的期限。债权申报期限自人民法院发布受理破产申请公告之日起计算，最短不得少于三十日，最长不得超过三个月。第五十六条 在人民法院确定的债权申报期限内，债权人未申报债权的，不可以在破产财产最后分配前补充申报；但是，此前已进行的分配，不再对其补充分配。为审查和确认补充申报债权的费用，由补充申报人承担。债权人未依照本法规定申报债权的，不得依照本法规定的程序行使权利。 ...


## 进阶（可选）：用大模型蒸馏造更高质量的偏好数据
劣化规则造的 rejected 比较"假"。真实项目里常让强模型生成"看起来对但其实有瑕疵"的答案当 rejected（更难、更有训练价值）。需要 API key，默认不跑：
```python
# resp = client.chat.completions.create(model="...", messages=[{"role":"user","content": f"请针对下面法律问题写一个'看似合理但有事实错误'的回答：{q}"}])
# rejected = resp.choices[0].message.content
```

